In [2]:
from collections import defaultdict, deque

def topo_sort(grafo, nodos):
    visitado = set()
    orden = []

    def dfs(u):
        pila = [(u, iter(grafo[u]))]
        visitado.add(u)
        while pila:
            nodo, vecinos = pila[-1]
            try:
                v, _ = next(vecinos)
                if v not in visitado:
                    visitado.add(v)
                    pila.append((v, iter(grafo[v])))
            except StopIteration:
                orden.append(pila.pop()[0])

    for n in nodos:
        if n not in visitado:
            dfs(n)
    return orden[::-1]



def dag_shortest_path(grafo, nodos, origen, destino = None):
    INF = float('inf')
    dist = {n: INF for n in nodos}
    prev = {n: None for n in nodos}
    dist[origen] = 0

    for u in topo_sort(grafo, nodos):
        if dist[u] == INF:
            continue
        for v, w in grafo[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u

    return dist, prev



def reconstruir_camino(prev, origen, destino):
    camino = []
    nodo = destino
    while nodo is not None:
        camino.append(nodo)
        nodo = prev[nodo]
    camino.reverse()
    return camino if camino[0] == origen else []



grafo = defaultdict(list, {
    'S': [('A', 2), ('B', 5), ('D', 3)],
    'A': [('C', 1), ('D', 2)],
    'B': [('D', 1), ('C', 3)],
    'C': [('E', -1)],
    'D': [('C', 2), ('E', 4)],
    'E': [('T', 2)],
    'T': [],
})
nodos = ['S', 'A', 'B', 'C', 'D', 'E', 'T']


dist, prev = dag_shortest_path(grafo, nodos, 'S')
camino = reconstruir_camino(prev, 'S','T')

print("Distancias: ", dist)

print("camino a T:", camino)
print("coste:", dist['T'])


def dag_longest_path(grafo, nodos, origen):
    NEG_INF = float('-inf')
    dist = {n: NEG_INF for n in nodos}
    dist[origen] = 0
    for u in topo_sort(grafo, nodos):
        if dist[u] == NEG_INF: continue
        for v, w in grafo[u]:
            if dist[u] + w > dist[v]:
                dist[v] = dist[u] + w

    return dist

Distancias:  {'S': 0, 'A': 2, 'B': 5, 'C': 3, 'D': 3, 'E': 2, 'T': 4}
camino a T: ['S', 'A', 'C', 'E', 'T']
coste: 4


In [3]:
def lis_dp(arr):
    n = len(arr)
    if not n: return[], 0

    dp = [1] * n
    prev = [-1]*n

    for i in range(1,n):
        for j in range(i):
            if arr[j] < arr[i] and dp[j] + 1> dp[i]:
                dp[i] = dp[j] + 1
                prev[i] = j

    end = max(range(n), key=lambda i: dp[i])

    lis, i = [], end
    while i != -1:
        lis.append(arr[i])
        i = prev[i]
    lis.reverse()

    return lis, dp[end]



arr = [10,9,8,3,5,6,7,188,97]
lis, length = lis_dp(arr)
print(lis)
print(length)



[3, 5, 6, 7, 188]
5


In [5]:
def distance_edicion(str1, str2):
    m = len(str1)
    n = len(str2)

    tabla = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        tabla[i][0] = i
    for j in range( n + 1):
        tabla[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # It should be str1[i - 1] and not str[i - 1]
            if str1[i - 1] == str2[j - 1]:
                tabla[i][j] = tabla[i - 1][j - 1]
            else:
              tabla[i][j] = 1 + min(
                  tabla[i - 1][j],
                  tabla[i][j - 1],
                  tabla[i - 1][j - 1]
              )

    return tabla[m][n]


palabra = "gato"
palabra2 = "casa"
resultado = distance_edicion(palabra, palabra2)

print(f"la distancia de edicion entre '{palabra}' y '{palabra2}' es: {resultado}")

la distancia de edicion entre 'gato' y 'casa' es: 3


In [7]:
def knapsack(valores, pesos, capacidad):
    n = len(valores)
    tabla = [[0] * (capacidad + 1) for _ in range(n + 1)]

    for i  in range(1, n + 1):
        for w in range(1, capacidad + 1):
            peso_actual = pesos[i - 1]
            valor_actual = valores[i - 1]


            if peso_actual <= w:
                tabla[i][w] = max(tabla[i - 1][w], valor_actual + tabla[i - 1][w - peso_actual])
            else:
                tabla[i][w] = tabla[i - 1][w]

    return tabla[n][capacidad]



valores = [60, 100, 120, 44]
pesos = [10,34,26, 20]
capacidad = 50

resultado = knapsack(valores, pesos, capacidad)
print(f"el valor maximo es: {resultado}")

el valor maximo es: 180


In [10]:
def knapsack_repeticion(valores, pesos, capacidad):
    dp = [0] * (capacidad + 1)

    for w in range(1, capacidad + 1):
        for i in range(len(valores)):
            if i < len(pesos) and pesos[i] <= w: # Added a check for list index out of range
                dp[w] = max(dp[w], dp[w - pesos[i]] + valores[i])

    return dp[capacidad]



valores = [30, 23, 45, 19]
pesos = [6 , 3, 4, 0] # Added a dummy weight to match the length of valores
capacidad = 10

resultado = knapsack_repeticion(valores, pesos, capacidad)
print(f"el valor con repeticiones: {resultado}")

el valor con repeticiones: 148


In [12]:
def knapsack_sin_repeticion(valores, pesos, capacidad):
    dp = [0] * (capacidad + 1)

    for i in range(len(valores)):
        for w in range(capacidad, pesos[i] - 1, -1):
            dp[w] = max(dp[w], dp[w - pesos[i]] + valores[i])
    return dp[capacidad]



valores = [60,100,120]
pesos = [10, 20, 30]
capacidad = 50

resultado = knapsack_sin_repeticion(valores, pesos, capacidad)
print(f"el valor maximo sin repeticion es: {resultado}")

el valor maximo sin repeticion es: 220


In [16]:
def matrix_chain_order(p):
    n = len(p) - 1

    m = [[0]* (n + 1) for _ in range(n + 1)]
    s = [[0]*(n +1) for _ in range(n + 1)]

    for l in range(2, n + 1):
        for i in range(1, n - l + 2):
            j = i + l - 1
            m[i][j] = float('inf')

            for k in range(i, j):
                costo = m[i][k] + m[k + 1][j] + p[i - 1]*p[k]*p[j]


                if costo < m[i][j]:
                    m[i][j] = costo
                    s[i][j] = k

    return m, s


def imprimir_pparentesis_optimo(s , i, j):
    if i == j:
        print(f"A{i}", end="")
    else:
        print("(", end="")
        imprimir_pparentesis_optimo(s, i , s[i][j])
        imprimir_pparentesis_optimo(s,s[i][j] + 1 , j)
        print(")", end="")



dimensiones = [10,20,30,40,30]

matriz_costos, cortes = matrix_chain_order(dimensiones)
n_matrices = len(dimensiones) - 1

print(f"el numero minimo de multiplicaciones escalares es: {matriz_costos[1][n_matrices]}")
imprimir_pparentesis_optimo(cortes, 1, n_matrices)
print()


el numero minimo de multiplicaciones escalares es: 30000
(((A1A2)A3)A4)
